# Iris Flower Classification

**Objective:** Train a machine learning classification model to identify the species of an iris flower
(*Setosa*, *Versicolor*, or *Virginica*) from its physical measurements (sepal length/width, petal length/width).

**Tech stack:** Python, scikit-learn, pandas, matplotlib, seaborn

**Dataset:** Built into scikit-learn (`sklearn.datasets.load_iris`) — no external download required.

---

## Table of Contents
1. [Setup & Imports](#1)
2. [Load Data](#2)
3. [Exploratory Data Analysis (EDA)](#3)
4. [Visualisations](#4)
5. [Feature Selection Discussion](#5)
6. [Train / Test Split](#6)
7. [Model Training](#7)
8. [Model Evaluation](#8)
9. [Best Model Selection](#9)
10. [Conclusion](#10)


<a id='1'></a>
## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

# Notebook display settings
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
RANDOM_STATE = 42

pd.set_option("display.max_columns", None)


<a id='2'></a>
## 2. Load Data

The Iris dataset ships with scikit-learn, so no download or external file is needed.

In [ ]:
iris = load_iris()

# Build a tidy DataFrame: features + human-readable species label
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["species_code"] = iris.target
df["species"] = df["species_code"].map(dict(enumerate(iris.target_names)))

df.head()


<a id='3'></a>
## 3. Exploratory Data Analysis (EDA)

We check the shape, data types, missing values, class balance, and descriptive statistics.

In [ ]:
# Shape of the dataset
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")


In [ ]:
# Data types
df.dtypes


In [ ]:
# Null / missing value check
print("Missing values per column:")
df.isnull().sum()


In [ ]:
# Class balance -- is the dataset balanced across species?
df["species"].value_counts()


In [ ]:
# Descriptive statistics for the numeric features
df.describe()


In [ ]:
# Descriptive statistics broken down by species
df.groupby("species").describe().T


**EDA takeaways:**
- The dataset has 150 rows and no missing values, so no imputation or cleaning is required.
- All three species (`setosa`, `versicolor`, `virginica`) are perfectly balanced with 50 samples each.
- Petal measurements show much larger spread between species than sepal measurements, hinting they will be strong predictors (explored further in Section 5).

<a id='4'></a>
## 4. Visualisations

In [ ]:
# Pairplot: feature distributions and pairwise relationships, coloured by species
sns.pairplot(df.drop(columns=["species_code"]), hue="species", diag_kind="hist", corner=True)
plt.suptitle("Pairwise Feature Relationships by Species", y=1.02)
plt.show()


In [ ]:
# Box plots for each feature, split by species
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
features = iris.feature_names

for ax, feature in zip(axes.flatten(), features):
    sns.boxplot(data=df, x="species", y=feature, ax=ax, hue="species", legend=False)
    ax.set_title(f"{feature} by species")

plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap between numeric features
plt.figure(figsize=(6, 5))
sns.heatmap(df[iris.feature_names].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Feature Correlation Heatmap")
plt.show()


<a id='5'></a>
## 5. Feature Selection Discussion

Which features are most discriminative between species?

In [ ]:
# Quantify separability: variance of each feature's per-species mean,
# relative to its overall spread. Higher = more discriminative.
summary = df.groupby("species")[iris.feature_names].mean()
discriminative_power = summary.var().sort_values(ascending=False)
discriminative_power


**Discussion:**
- The box plots and pairplot above show that **petal length** and **petal width** separate the three species almost perfectly, with very little overlap between classes.
- **Sepal length** and especially **sepal width** overlap heavily between `versicolor` and `virginica`, making them weaker standalone predictors.
- The correlation heatmap confirms petal length and petal width are strongly correlated with each other (both track flower size/maturity), while sepal width is only weakly correlated with the rest.
- **Conclusion:** all four features are kept for modelling (the dataset is small and low-dimensional, so there is no need to drop features), but we expect petal measurements to dominate feature importance in tree-based models.

<a id='6'></a>
## 6. Train / Test Split

An 80/20 split, stratified by species so both sets keep the same class balance.

In [ ]:
X = df[iris.feature_names]
y = df["species_code"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set:  {X_test.shape[0]} samples")


In [ ]:
# Standardise features (helps distance-based / gradient-based models like KNN & Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


<a id='7'></a>
## 7. Model Training

We train **four** classifiers so we can compare a linear model, a distance-based model, and two tree-based models:
1. Logistic Regression
2. K-Nearest Neighbours
3. Decision Tree
4. Random Forest

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=200, random_state=RANDOM_STATE),
    "K-Nearest Neighbours": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
}

# Logistic Regression & KNN use scaled features; tree-based models are scale-invariant
scaled_models = {"Logistic Regression", "K-Nearest Neighbours"}

trained_models = {}
for name, model in models.items():
    if name in scaled_models:
        model.fit(X_train_scaled, y_train)
    else:
        model.fit(X_train, y_train)
    trained_models[name] = model

print("All models trained.")


<a id='8'></a>
## 8. Model Evaluation

For each model: accuracy, confusion matrix, and a full classification report (precision, recall, F1).

In [ ]:
results = []

for name, model in trained_models.items():
    X_eval = X_test_scaled if name in scaled_models else X_test
    y_pred = model.predict(X_eval)

    acc = accuracy_score(y_test, y_pred)
    results.append({"Model": name, "Accuracy": acc})

    print("=" * 60)
    print(f"{name}  |  Accuracy: {acc:.4f}")
    print("=" * 60)
    print(classification_report(y_test, y_pred, target_names=iris.target_names))


In [ ]:
# Confusion matrices for every model, side by side
fig, axes = plt.subplots(1, len(trained_models), figsize=(20, 4.5))

for ax, (name, model) in zip(axes, trained_models.items()):
    X_eval = X_test_scaled if name in scaled_models else X_test
    y_pred = model.predict(X_eval)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=iris.target_names)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name)

plt.tight_layout()
plt.show()


In [ ]:
# Summary comparison table
results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False).reset_index(drop=True)
results_df


In [ ]:
# Feature importance from the Random Forest -- confirms the Section 5 discussion
importances = pd.Series(
    trained_models["Random Forest"].feature_importances_,
    index=iris.feature_names,
).sort_values(ascending=False)

plt.figure(figsize=(7, 4))
sns.barplot(x=importances.values, y=importances.index, hue=importances.index, legend=False, palette="viridis")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.show()

importances


<a id='9'></a>
## 9. Best Model Selection

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_accuracy = results_df.iloc[0]["Accuracy"]

print(f"Best-performing model: {best_model_name}")
print(f"Test accuracy: {best_accuracy:.4f}")


**Justification:**
On this dataset, several models typically reach very high (often 100%) accuracy on the 30-sample test set because the classes — especially `setosa` — are almost linearly separable using petal measurements alone. Among models that tie on accuracy, the tie-breakers are:
- **Simplicity / interpretability:** Logistic Regression and Decision Tree are easy to explain to a non-technical audience.
- **Robustness to overfitting:** Random Forest averages many trees, so it tends to generalise better than a single Decision Tree on data it hasn't seen, even if their test-set accuracy on this small dataset looks identical.
- **Confusion matrix:** the chosen model should make zero or the fewest mistakes between `versicolor` and `virginica`, the two species that overlap most.

Given the balance of accuracy, robustness, and the confusion matrix results above, the **best-performing model** printed above is selected as the final model for this task. (Re-run the notebook and inspect `results_df` — the printed name will reflect the actual run.)

<a id='10'></a>
## 10. Conclusion

- The Iris dataset is clean, balanced, and required no external sourcing or preprocessing beyond scaling.
- Petal length and petal width are the most discriminative features; sepal width is the weakest.
- Multiple classifiers were trained and evaluated with accuracy, confusion matrices, and classification reports.
- The best-performing model (selected above) is a strong, well-justified choice for this classification task.

**Possible next steps:** cross-validation, hyperparameter tuning (e.g. `GridSearchCV`), and trying additional models such as SVM or Gradient Boosting.